In [0]:
from pyspark.sql.types import StringType, StructField, StructType

import csv, io

In [0]:
customer_data = """source_system,source_cust_id,first_name,last_name,email,phone,postal_code
POS,POS-10001,Katherine,Miller,kmiller12@gmail.com,(603) 555-0143,03101
WEB,WEB-20001,Kathy,Miller,kmiller12@gmail.com,6035550143,03101-1204
CATALOG,CAT-30001,K.,Miller,,603-555-0143,03101
POS,POS-10002,David,Miller,dmiller99@yahoo.com,(603) 555-0188,03101
WEB,WEB-20002,Dave,Miller,dmiller99@yahoo.com,6035550188,03101-1204
POS,POS-10003,Margaret,Sullivan,msullivan@comcast.net,+1(617) 555-2341,02116
CATALOG,CAT-30002,Peggy,Sullivan,,6175552341,02116
WEB,WEB-20003,Margaret,Sullivan,msullivan@comcast.net ,617-555-2341,02116-3301
POS,POS-10004,Elizabeth,Baker,ebaker@outlook.com ,(401) 555-9081,02903
WEB,WEB-20004,Beth,Baker,ebaker@outlook.com,4015559081,02903
CATALOG,CAT-30003,E.,Baker,liz.baker@verizon.net,401-555-9081,02903-4412
POS,POS-10005,Sarah,Jenkins,sjenkins@icloud.com,+1(207) 555-7762,04101
WEB,WEB-20005,Sara,Jenkins,sjenkins@icloud.com,2075557762,04101
POS,POS-10006,Robert,Jenkins,rjenkins_nh@gmail.com,(207) 555-7769,04101
CATALOG,CAT-30004,Bob,Jenkins, rjenkins_nh@gmail.com,207-555-7769,04101-2210
WEB,WEB-20006,Jennifer,Walsh,jwalsh88@hotmail.com,5085554320,01701
POS,POS-10007,Jenn,Walsh,jwalsh88@hotmail.com,+91(508) 555-4320,01701
CATALOG,CAT-30005,J.,Walsh,,508-555-4320,01701
WEB,WEB-20007,Patricia,O'Connor,poconnor@gmail.com,6035558911,03801
POS,POS-10008,Pat,O'Connor,poconnor@gmail.com,(603) 555-8911,03801-4410"""

# Build schema from row 1 (the header row) — all columns as StringType
# columns = customer_data.split("\n")[0].split(",")
columns = list(csv.reader(io.StringIO(customer_data)))[0]
schema = StructType([StructField(c, StringType(), True) for c in columns])

# Data rows are everything after row 1
rows = [line.split(",") for line in customer_data.split("\n")[1:]]

# Empty fields (e.g., missing email) become NULL
rows = [[None if v == "" else v for v in r] for r in rows]

customer_df = spark.createDataFrame(rows, schema)
display(customer_df)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import *

customer_df = customer_df\
        .withColumn("clean_phone", F.regexp_replace(F.col("phone"), r"[^0-9]", ""))\
            .withColumn("len_phose", F.length(F.col("clean_phone")))\
                .withColumn("original_number", F.right(F.col("clean_phone"), F.lit(10)))\
                    .withColumn("clean_email", F.trim(F.col("email")))

customer_df.createOrReplaceTempView("customer")

In [0]:
%pip install rapidfuzz
import pandas as pd
from pyspark.sql.functions import pandas_udf
from rapidfuzz.distance import JaroWinkler

# Vectorized UDF handles processing over Apache Arrow batches smoothly
@pandas_udf("double")
def jaro_winkler_sim(s1: pd.Series, s2: pd.Series) -> pd.Series:
    # rapidfuzz returns a value between 0 and 100, normalize it to 0.0 - 1.0
    return pd.Series([JaroWinkler.similarity(a, b) / 100.0 for a, b in zip(s1, s2)])

# Register it to use inside Databricks SQL notebooks
spark.udf.register("jaro_winkler_sim", jaro_winkler_sim)

In [0]:
df = spark.sql("""
               select *,case when jaro_winkler_sim()
               from customer
               """)

In [0]:

address_data = """source_id,source_cust_id,raw_street,raw_city,raw_state,raw_zip
ADDR-SRC-01,POS-10001,142 Elm Street Apt 2B,Manchester,NH,03101
ADDR-SRC-02,WEB-20001,142 Elm St # 2B,MANCHESTER,New Hampshire,03101-1204
ADDR-SRC-03,CAT-30001,142 Elm St.,Manchester,NH,03101
ADDR-SRC-04,POS-10002,142 Elm St. Suite 2B,Manchester,NH,03101
ADDR-SRC-05,WEB-20002,142 Elm St Apt 2-B,Manchester,NH,03101-1204
ADDR-SRC-06,POS-10003,77 Commonwealth Ave Floor 3,Boston,MA,02116
ADDR-SRC-07,CAT-30002,77 Commonwealth Avenue Apt 3,Boston,MA,02116
ADDR-SRC-08,WEB-20003,77 Commonwealth Ave #3,BOSTON,Massachusetts,02116-3301
ADDR-SRC-09,POS-10004,12 Hope Street,Providence,RI,02903
ADDR-SRC-10,WEB-20004,12 Hope St,Providence,Rhode Island,02903
ADDR-SRC-11,CAT-30003,12 Hope St.,Providence,RI,02903-4412
ADDR-SRC-12,POS-10005,450 Ocean Avenue,Portland,ME,04101
ADDR-SRC-13,WEB-20005,450 Ocean Ave,PORTLAND,Maine,04101
ADDR-SRC-14,POS-10006,450 Ocean Avenue,Portland,ME,04101
ADDR-SRC-15,CAT-30004,450 Ocean Ave.,Portland,ME,04101-2210
ADDR-SRC-16,WEB-20006,19 Maple Road,Framingham,MA,01701
ADDR-SRC-17,POS-10007,19 Maple Rd,Framingham,Massachusetts,01701
ADDR-SRC-18,CAT-30005,19 Maple Road,Framingham,MA,01701
ADDR-SRC-19,WEB-20007,88 Market Street Unit 10,Portsmouth,NH,03801
ADDR-SRC-20,POS-10008,88 Market St #10,Portsmouth,NH,03801-4410
"""

# columns = address_data.split("\n")[0].split(",")
columns = list(csv.reader(io.StringIO(address_data)))[0]
schema = StructType([StructField(c, StringType(), True) for c in columns])

display(columns)
display(schema)

rows = list(csv.reader(io.StringIO(address_data)))[1:]

address_df = spark.createDataFrame(rows, schema)
display(address_df)

In [0]:
items_data = """source_system,source_sku_id,source_description,upc,color_code,size_code
Oracle EBS,EBS-1001-NVY-S,PIMA TEE - NAVY - SM,088341901011,410,SM
PIM,PIM-SKU-1001,"Pure Jill Pima Tee, Classic Navy, Small",088341901011,NVY,S
POS,POS-ITM-1001,PJ PIMA TEE CLS NAV S,,410,S
Oracle EBS,EBS-1002-NVY-M,PIMA TEE - NAVY - MED,088341901028,410,MED
PIM,PIM-SKU-1002,"Pure Jill Pima Tee, Classic Navy, Medium",088341901028,NVY,M
POS,POS-ITM-1002,PJ PIMA TEE CLS NAV M,,410,M
Oracle EBS,EBS-1003-WHT-M,PIMA TEE - WHITE - MED,088341901035,100,MED
PIM,PIM-SKU-1003,"Pure Jill Pima Tee, White, Medium",088341901035,WHT,M
POS,POS-ITM-1003,PJ PIMA TEE WHT M,,100,M
Oracle EBS,EBS-2001-KHK-06,LINEN ANKLE PANT - KHAKI - 6,088341902018,220,06
PIM,PIM-SKU-2001,"Everyday Linen Pants, Flax Khaki, 6",088341902018,KHK,6
POS,POS-ITM-2001,ED LINEN PANT KHK 06,,220,6
Oracle EBS,EBS-2002-BLK-08,LINEN ANKLE PANT - BLACK - 8,088341902025,001,08
PIM,PIM-SKU-2002,"Everyday Linen Pants, Black, 8",088341902025,BLK,8
POS,POS-ITM-2002,ED LINEN PANT BLK 08,,001,8
Oracle EBS,EBS-3001-CHR-L,CASHMERE CARDI - CHARCOAL - LG,088341903015,015,LG
PIM,PIM-SKU-3001,"Luxe Cashmere Open Cardigan, Heather Charcoal, Large",088341903015,CHR,L
POS,POS-ITM-3001,LX CASH CARDI CHR L,,015,L
Oracle EBS,EBS-4001-IND-10,STRETCH SLIM DENIM - INDIGO - 10,088341904012,450,10
PIM,PIM-SKU-4001,"Authentic Stretch Denim, Deep Indigo, 10",088341904012,IND,10"""

columns = items_data.split("\n")[0].split(',')
schema = StructType([StructField(c, StringType(), True) for c in columns])

import csv, io
rows = list(csv.reader(io.StringIO(items_data)))[1:]
# display(rows)

items_data = spark.createDataFrame(rows, schema)
display(items_data)